In [19]:
from pathlib import Path
from matplotlib import pyplot as plt
import warnings
import prism
import pandas as pd
import pickle
from pathlib import Path 
from imagematerials.reporting.reporting import create_iamc_reporting, create_iamc_eol, export_iamc_csv
from imagematerials.reporting import iamc_config as CFG
from types import SimpleNamespace
import xarray as xr
import numpy as np 

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.buildings.preprocessing.floorspace import get_image_floorspace, get_floorspace_urban_rural
from imagematerials.buildings.preprocessing.population import compute_population

### define scenarios and import pickle

In [6]:
DESIRED_SCENARIOS = [
    "SSP2", 
    "SSP2_narrow_act",
    "SSP2_narrow_prod",
    "SSP2_narrow",
    "SSP2_slow",
    "SSP2_close",
    "SSP2_narrow_slow_close",

    "SSP2_26_tax",
    "SSP2_narrow_act_26_tax",
    'SSP2_narrow_prod_26_tax',
    'SSP2_narrow_26_tax',
    "SSP2_slow_26_tax",
    "SSP2_close_26_tax",
    "SSP2_narrow_slow_close_26_tax",

    "SSP2_19_tax",
    "SSP2_narrow_slow_close_19_tax",
    ]  

pkl_path = Path("..","data", "output","ALL_OUTPUT_ALL_SCENARIOS_14-4-26.pkl")

with open(pkl_path, "rb") as f:
    all_output = pickle.load(f)

outdir = Path('data','output','reporting','final')

### sectoral reporting

#### vehicles

In [13]:
output_vhc = {
    iamc_label: SimpleNamespace(
        vehicles={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][0],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][0],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][0],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][0],
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][0],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][0],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

# Vehicles reporting
df_vhc = create_iamc_reporting(
    models=output_vhc,
    templates=[
    "Final Product Demand|Transportation|{Vehicle Types}",
    "Product Stock|Transportation|{Vehicle Types}",
    "Stock Retirement|Transportation|{Vehicle Types}",
    "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}",
    "Material Stock|{Engineered Material}|{Demand Sector Disaggregated}",
    "Material Outflow|{Engineered Material}|{Demand Sector Disaggregated}",
    ],
    sector="vehicles",               
    model_name="IMAGE 3.4",
    outdir=outdir
)

#### buildings

In [14]:
output_bld = {
    iamc_label: SimpleNamespace(
        buildings={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][1],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][1],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][1],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][1],
            "stock_by_cohort":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],  # Map stocks to stock_by_cohort
            "outflow_by_cohort": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],  # Map outflow to outflow_by_cohort
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

# Buildings reporting
df_bld = create_iamc_reporting(
    models = output_bld,
    templates=[
        "Final Product Demand|Buildings|{Building Types}",
        "Product Stock|Buildings|{Building Types}",
        "Stock Retirement|Buildings|{Building Types}",
        "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}",
        "Material Stock|{Engineered Material}|{Demand Sector Disaggregated}",
        "Material Outflow|{Engineered Material}|{Demand Sector Disaggregated}",
    ],
    sector = "buildings",
    model_name="IMAGE 3.4",
    outdir=outdir
)


### single variable reporting

In [11]:
# Define the scenario list
scenario_list = {
                
                "base":             ("SSP2",["base"]),
                "narrow_product":   ("SSP2_narrow_prod", ["base", "narrow_product"]),
                "narrow_activity":  ("SSP2_narrow_act", ["base", "narrow_activity"]),
                "narrow":           ("SSP2_narrow", ["base", "narrow_product", "narrow_activity"]),
                "slow":             ("SSP2_slow",["base", "slow"]),
                "close":            ("SSP2_close",["base", "close"]),
                "narrow_slow_close":("SSP2_narrow_slow_close", ["base", "narrow_product", "narrow_activity","slow","close"]),

                 "base_26":             ("SSP2_26_tax", ["base"]),
                 "narrow_product_26":   ("SSP2_narrow_prod_26_tax", ["base", "narrow_product"]),
                 "narrow_activity_26":  ("SSP2_narrow_act_26_tax", ["base", "narrow_activity"]),
                 "narrow_26":           ("SSP2_narrow_26_tax", ["base", "narrow_product", "narrow_activity"]),
                "slow_26":             ("SSP2_slow_26_tax",["base", "slow"]),
                "close_26":            ("SSP2_close_26_tax",["base", "close"]),
                 "narrow_slow_close_26":("SSP2_narrow_slow_close_26_tax", ["base", "narrow_product", "narrow_activity","slow","close"]),

                "base_19":             ("SSP2_19_tax", ["base"]),
                "narrow_slow_close_19":("SSP2_narrow_slow_close_19_tax", ["base", "narrow_product", "narrow_activity","slow","close"]),
                }

# Define paths
scenario_base_path = Path("..", "data", "raw")
CP_scenario_path = scenario_base_path/ "IMAGE_CircoMod"
CE_scenario_path = scenario_base_path / 'circular_economy_scenarios'

materials_dict = {
    "steel": "Steel",
    "aluminium": "Aluminum",  
    "concrete": "Concrete",
    "copper": "Copper",
}

#### checking results

In [ ]:
print("--------------------------what's in the pickle?-------------------------")

scenarios = all_output.keys()
vars = all_output["base"].keys()
materials = all_output["base"]["inflow_materials"][0].to_array().coords["material"].values
years = all_output["base"]["inflow_materials"][0].to_array().coords["time"].values

print(f"available scenarios:{scenarios}")
print(f"available variables:{vars}" )
print(f"available materials: {materials}")
print(f"available years: {years}")
print("-------------------------------------------------------------------------")
print("sectors for common variables: ")
print("[0]: vehicles")
print("[1]: buildings")
print("[2]: electricity_generation")
print("[3,4]: electricity_T&D")
print("[5,6]: electricity_storage")
print("[7]: rest-of")      
print("common variables = inflow_materials")
print("common variables (except rest-of): stock_by_cohort_materials, outflow_by_cohort_materials, inflow, stocks, outflow")
print("eol variables = sum_outflow, reusable_materials, recyclable_materials")

print("-------------------------------------------------------------------------")



--------------------------what's in the pickle?-------------------------
available scenarios:dict_keys(['base', 'narrow_product', 'narrow_activity', 'narrow', 'slow', 'close', 'narrow_slow_close', 'base_26', 'narrow_product_26', 'narrow_activity_26', 'narrow_26', 'slow_26', 'close_26', 'narrow_slow_close_26', 'base_19', 'narrow_slow_close_19'])
available variables:dict_keys(['inflow_materials', 'stock_by_cohort_materials', 'outflow_by_cohort_materials', 'inflow', 'stocks', 'outflow', 'sum_outflow', 'reusable_materials', 'recyclable_materials'])
available materials: ['aluminium' 'cobalt' 'copper' 'glass' 'lead' 'lithium' 'manganese'
 'neodymium' 'nickel' 'plastics' 'rubber' 'steel' 'titanium' 'wood']
available years: [1971 1972 1973 1974 1975 1976 1977 1978 1979 1980 1981 1982 1983 1984
 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997 1998
 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012
 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 

In [17]:
# examples
# you can print and check w/ the data generated in csv
da = all_output["base"]["inflow_materials"][0].to_array().sel(material = "steel", Region = "CHN").sum("Type")

da2 = all_output["base"]["sum_outflow"].to_array().sel(material = "steel", Region = "CHN").sum("Type")

#### end-of-life

In [21]:
# ---------"Scrap|...|...|{Demand Sector}"----------------

# sectoral scrap generated (before collection for reuse and recycling)
# totals will not be reported yet, we need to add transportation infrastructure for consistency

types = {
"building_types": ['commercial','rural',  'urban'],
"vehicle_types": ['freight', 'passenger'],
"electricity_types": [ 'generation', 'grid',  'storage']
}

demand_sector = {
    "building_types": "Buildings",
    "vehicle_types": "Transportation",
    "electricity_types": "Electricity"
}

# steel
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    arr = all_output[scen_id]["sum_outflow"].to_array()

    for group_name, subtype_list in types.items():
        sector_label = demand_sector[group_name]

        da = (
            arr.sel(material="steel", Type=subtype_list)  
                .sum(dim="Type")                       
        )
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Scrap|Iron and Steel|Steel|{sector_label}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

# aluminium
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    arr = all_output[scen_id]["sum_outflow"].to_array()

    for group_name, subtype_list in types.items():
        sector_label = demand_sector[group_name]

        da = (
            arr.sel(material="aluminium", Type=subtype_list)  
                .sum(dim="Type")                       
        )
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Scrap|Non-Ferrous Metals|Aluminum|{sector_label}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )



Running reporting for: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_prod from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']
Running reporting for: SSP2_narrow_act from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_activity']
Running reporting for: SSP2_narrow from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity']
Running reporting for: SSP2_slow from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'slow']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_close from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'close']
Running reporting for: SSP2_narrow_slow_close from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']
Running reporting for: SSP2_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
Running reporting for: SSP2_narrow_prod_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']


...

c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_prod_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']
Running reporting for: SSP2_narrow_act_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_activity']
Running reporting for: SSP2_narrow_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity']
Running reporting for: SSP2_slow_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'slow']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_close_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'close']
Running reporting for: SSP2_narrow_slow_close_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']
Running reporting for: SSP2_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
Running reporting for: SSP2_narrow_slow_close_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

In [22]:


# ---------"Material Outflow Potential|Secondary Material from Reuse|{Engineered Material}"----------------

# total reusable scrap scrap generated after reuse rate
# we need to add transportation infrastructure to these values.

for scen_id, (climate_scen, circular_scen) in scenario_list.items():

    arr = all_output[scen_id]["reusable_materials"].to_array()

    for mat, mat_dict in materials_dict.items():
        da = (
            arr.sel(material=mat)  
                .sum(dim="Type")                       
        )
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Outflow Potential|Secondary Material from Reuse|{mat_dict}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )


# ---------"Material Outflow Potential|Secondary Material from Recycling|{Engineered Material}"----------------

# total reusable scrap scrap generated after reuse rate
# we need to add transportation infrastructure to these values.


for scen_id, (climate_scen, circular_scen) in scenario_list.items():

    arr = all_output[scen_id]["recyclable_materials"].to_array()

    for mat, mat_dict in materials_dict.items():
        da = (
            arr.sel(material=mat)  
                .sum(dim="Type")                       
        )
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Outflow Potential|Secondary Material from Recycling|{mat_dict}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

#### electricity

In [23]:
# --------------------------------------------------GENERATION -------------------------------------------- 

#            FINAL MATERIAL DEMAND

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    arr = all_output[scen_id]["inflow_materials"][2].to_array()

    for mat, mat_dict in materials_dict.items():
        da = arr.sel(material=mat).sum(dim="Type")

        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Final Material Demand|{mat_dict}|Electricity|Generation",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

#            MATERIAL STOCK


for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    arr = all_output[scen_id]["stock_by_cohort_materials"][2]

    for mat, mat_dict in materials_dict.items():
        da = arr.sel(material=mat).sum(dim="Type")

        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Stock|{mat_dict}|Electricity|Generation",
            factor=1e-9,
            unit="Mt",
            outdir=outdir / climate_scen,
        )

#            MATERIAL OUTFLOW

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    arr = all_output[scen_id]["outflow_by_cohort_materials"][2].to_array()

    for mat, mat_dict in materials_dict.items():
        da = arr.sel(material=mat).sum(dim="Type")

        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Outflow|{mat_dict}|Electricity|Generation",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

Running reporting for: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
Running reporting for: SSP2_narrow_prod from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']
Running reporting for: SSP2_narrow_act from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_activity']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity']
Running reporting for: SSP2_slow from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'slow']
Running reporting for: SSP2_close from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'close']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_slow_close from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']
Running reporting for: SSP2_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
Running reporting for: SSP2_narrow_prod_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']


...

c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_close_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'close']
Running reporting for: SSP2_narrow_slow_close_26_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']
Running reporting for: SSP2_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_slow_close_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)


In [24]:
# -------------------------------------------------- TRANSMISSION + DISTRIBUTION --------------------------------------------
# sectors: Transmission = [3], Distribution = [4]
ELC_TD_LABEL = "Transmission and Distribution"

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    # ============================ FINAL MATERIAL DEMAND ============================
    da3_fmd = all_output[scen_id]["inflow_materials"][3].to_array().sum(dim="Type")
    da4_fmd = all_output[scen_id]["inflow_materials"][4].to_array().sum(dim="Type")
    da_td_fmd = da3_fmd + da4_fmd

    for mat, mat_dict in materials_dict.items():
        da = da_td_fmd.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Final Material Demand|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

    # ============================ MATERIAL STOCK ============================
    da3_stk = all_output[scen_id]["stock_by_cohort_materials"][3].sum(dim="Type")
    da4_stk = all_output[scen_id]["stock_by_cohort_materials"][4].sum(dim="Type")
    da_td_stk = da3_stk + da4_stk

    for mat, mat_dict in materials_dict.items():
        da = da_td_stk.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Stock|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt",
            outdir=outdir / climate_scen,
        )

    # ============================ MATERIAL OUTFLOW ============================
    da3_out = all_output[scen_id]["outflow_by_cohort_materials"][3].to_array().sum(dim="Type")
    da4_out = all_output[scen_id]["outflow_by_cohort_materials"][4].to_array().sum(dim="Type")
    da_td_out = da3_out + da4_out

    for mat, mat_dict in materials_dict.items():
        da = da_td_out.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Outflow|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )


Running reporting for: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_prod from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_act from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_activity']


...

c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_slow_close_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)


In [25]:
# -------------------------------------------------- STORAGE --------------------------------------------
# sectors: PHS = [5], oTHER = [6]
ELC_TD_LABEL = "Storage"

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running reporting for: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")

    # ============================ FINAL MATERIAL DEMAND ============================
    da5_fmd = all_output[scen_id]["inflow_materials"][5].to_array().sum(dim="Type")
    da6_fmd = all_output[scen_id]["inflow_materials"][6].to_array().sum(dim="Type")
    da_td_fmd = da5_fmd + da6_fmd

    for mat, mat_dict in materials_dict.items():
        da = da_td_fmd.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Final Material Demand|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )

    # ============================ MATERIAL STOCK ============================
    da5_stk = all_output[scen_id]["stock_by_cohort_materials"][5].sum(dim="Type")
    da6_stk = all_output[scen_id]["stock_by_cohort_materials"][6].sum(dim="Type")
    da_td_stk = da5_fmd + da6_fmd

    for mat, mat_dict in materials_dict.items():
        da = da_td_stk.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Stock|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt",
            outdir=outdir / climate_scen,
        )

    # ============================ MATERIAL OUTFLOW ============================
    da5_out = all_output[scen_id]["outflow_by_cohort_materials"][5].to_array().sum(dim="Type")
    da6_out = all_output[scen_id]["outflow_by_cohort_materials"][6].to_array().sum(dim="Type")
    da_td_out = da5_fmd + da6_fmd

    for mat, mat_dict in materials_dict.items():
        da = da_td_out.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Material Outflow|{mat_dict}|Electricity|{ELC_TD_LABEL}",
            factor=1e-9,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )


Running reporting for: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_prod from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_act from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_activity']
Running reporting for: SSP2_narrow from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity']


...

c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

Running reporting for: SSP2_narrow_slow_close_19_tax from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base', 'narrow_product', 'narrow_activity', 'slow', 'close']


c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Zanon004\AppData\Local\miniforge3\envs\mat-env2\Lib\site-packages\numpy\lib\_function_base_impl.py:2616: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\U

#### total material demands

In [26]:
# -------------------------------------------------- TOTAL MATERIAL DEMAND --------------------------------------------
# adding all sectors [0->7]
materials_dict2 = {
    "steel": "Steel",
    "aluminium": "Aluminum",  
    #"concrete": "Concrete",
    "copper": "Copper",
}

def to_Mt_plain(da):
    """
    Convert a pint-xarray DataArray to Mt (megatonnes) as plain float values.
    Works for kg, metric_ton, etc., as long as pint can convert to kilogram.
    """
    da_kg = da.pint.to("kilogram")          # convert units properly
    da_plain = da_kg.pint.dequantify()      # strip pint Quantity -> numpy floats
    return da_plain * 1e-9                  # kg -> Mt

def normalize_time(da):
    """
    Ensure the time dimension is always called 'time'.
    """
    if "Time" in da.dims:
        da = da.rename({"Time": "time"})
    return da

# all materials except cement
for scen_id, (climate_scen, circular_scen) in scenario_list.items():

    da0_fmd = normalize_time(all_output[scen_id]["inflow_materials"][0].to_array().sum(dim="Type")) # vehicles
    da1_fmd = normalize_time(all_output[scen_id]["inflow_materials"][1].to_array().sum(dim="Type")) # buildings
    da2_fmd = normalize_time(all_output[scen_id]["inflow_materials"][2].to_array().sum(dim="Type")) # electricity generation
    da3_fmd = normalize_time(all_output[scen_id]["inflow_materials"][3].to_array().sum(dim="Type")) # electricity transmission and distribution - lines
    da4_fmd = normalize_time(all_output[scen_id]["inflow_materials"][4].to_array().sum(dim="Type")) # electricity transmission and distribution - transformers and substations
    da5_fmd = normalize_time(all_output[scen_id]["inflow_materials"][5].to_array().sum(dim="Type")) # electricity storage - PHS
    da6_fmd = normalize_time(all_output[scen_id]["inflow_materials"][6].to_array().sum(dim="Type")) # electricity storage - Other
    da7_fmd = normalize_time(all_output[scen_id]["inflow_materials"][7])                            # rest-of 
    
    das_fmd = [da0_fmd, da1_fmd, da2_fmd, da3_fmd, da4_fmd, da5_fmd, da6_fmd, da7_fmd]
    
    # convert everything to Mt before summing
    das_fmd = [to_Mt_plain(d).fillna(0.0) for d in das_fmd]

   # build material list based on rest-of (these are the only materials the we can report totals; for other materials we only cover part of the demand, which is already reported per sector)
    master_mats = da7_fmd.coords["material"].values
    # reindex each sector to the union, fill missing materials with 0
    das_fmd = [
        (da_fmd.reindex(material=master_mats).fillna(0.0) if "material" in da_fmd.dims else da_fmd)
        for da_fmd in das_fmd
    ]
    # align on any other coords too (Region/Time), outer join, fill missing with 0
    das_fmd = xr.align(*das_fmd, join="outer", fill_value=0.0)

    # sum 
    da_total_fmd = sum(das_fmd)

    for mat, mat_dict in materials_dict2.items():
        da = da_total_fmd.sel(material=mat)
        export_iamc_csv(
            da,
            model="IMAGE 3.4",
            scenario=climate_scen,
            variable=f"Final Material Demand|{mat_dict}",
            factor=1,
            unit="Mt/yr",
            outdir=outdir / climate_scen,
        )


In [27]:
# -------------------------------------------------- TOTAL MATERIAL DEMAND (CEMENT) --------------------------------------------
# adding all sectors [1->7], vehicles doesn't have concrete as material coord
cement_in_concrete_factor = 0.12        # this value usually vary between 10-15%, we're using 12%
# for cement
for scen_id, (climate_scen, circular_scen) in scenario_list.items():

    #da0_fmd = all_output[scen_id]["inflow_materials"][0].to_array().sel(material = "concrete").sum(dim="Type") # vehicles
    da1_fmd = all_output[scen_id]["inflow_materials"][1].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # buildings
    da2_fmd = all_output[scen_id]["inflow_materials"][2].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # electricity generation
    da3_fmd = all_output[scen_id]["inflow_materials"][3].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # electricity transmission and distribution - lines
    da4_fmd = all_output[scen_id]["inflow_materials"][4].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # electricity transmission and distribution - transformers and substations
    da5_fmd = all_output[scen_id]["inflow_materials"][5].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # electricity storage - PHS
    da6_fmd = all_output[scen_id]["inflow_materials"][6].to_array().sel(material = "concrete").sum(dim="Type")*cement_in_concrete_factor # electricity storage - Other
    da7_fmd = all_output[scen_id]["inflow_materials"][7] # rest-of (there's already a cement coordinate)
    
    das_fmd = [da1_fmd, da2_fmd, da3_fmd, da4_fmd, da5_fmd, da6_fmd, da7_fmd]

    
    # convert everything to Mt before summing
    das_fmd = [to_Mt_plain(d).fillna(0.0) for d in das_fmd]

    # align on any other coords too (Region/Time), outer join, fill missing with 0
    das_fmd = xr.align(*das_fmd, join="outer", fill_value=0.0)

    # sum 
    da_total_fmd = sum(das_fmd)

    da = da_total_fmd.sel(material="cement")
    export_iamc_csv(
        da,
        model="IMAGE 3.4",
        scenario=climate_scen,
        variable=f"Final Material Demand|Cement",
        factor=1,
        unit="Mt/yr",
        outdir=outdir / climate_scen,
    )

#### multiple csvs in a scenario folder --> one csv per scenario

In [33]:

from __future__ import annotations

from pathlib import Path
import pandas as pd

# ----------------------------- CONFIG -----------------------------
ROOT_DIR = Path("..","..",'data','April 2026 runs','combine_new')

OUT_DIR = ROOT_DIR / "_updated_excels"
SHEET_NAME = "data"

EXCEL_EXTS = {".xlsx", ".xlsm", ".xls"}
CSV_EXTS = {".csv"}

# If True, try common fallbacks if CSV read fails
CSV_FALLBACKS = True

# If True, write a backup copy name even when overwriting is allowed
NEVER_OVERWRITE_ORIGINALS = True
# -----------------------------------------------------------------


def is_excel(p: Path) -> bool:
    return p.suffix.lower() in EXCEL_EXTS


def is_csv(p: Path) -> bool:
    return p.suffix.lower() in CSV_EXTS


def read_csv_robust(path: Path) -> pd.DataFrame | None:
    """
    Try reading CSV with sensible fallbacks (comma/semicolon, utf-8/latin1).
    """
    tries = [
        dict(sep=",", encoding="utf-8"),
        dict(sep=";", encoding="utf-8"),
        dict(sep=",", encoding="utf-8-sig"),
        dict(sep=";", encoding="utf-8-sig"),
    ]
    if CSV_FALLBACKS:
        tries += [
            dict(sep=",", encoding="latin1"),
            dict(sep=";", encoding="latin1"),
            dict(sep=None, engine="python", encoding="utf-8"),   # autodetect sep
            dict(sep=None, engine="python", encoding="latin1"),
        ]

    for kw in tries:
        try:
            df = pd.read_csv(path, **kw)
            # guard: if separator wrong, you'll often get 1 column with lots of separators in header
            if df.shape[1] == 1 and df.columns.size == 1 and ("," in str(df.columns[0]) or ";" in str(df.columns[0])):
                continue
            return df
        except Exception:
            continue

    print(f"[WARN] Failed reading CSV {path.name} with all strategies.")
    return None


def read_excel_sheet(excel_path: Path, sheet_name: str) -> pd.DataFrame | None:
    try:
        xls = pd.ExcelFile(excel_path)
        if sheet_name not in xls.sheet_names:
            return None
        return pd.read_excel(xls, sheet_name=sheet_name)
    except Exception as e:
        print(f"[WARN] Failed reading Excel {excel_path.name}: {e}")
        return None


def align_to_base(df: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """
    Align df to base_cols:
      - add missing base columns as NA
      - drop extra columns
      - reorder
    """
    out = df.copy()
    for c in base_cols:
        if c not in out.columns:
            out[c] = pd.NA
    out = out[base_cols]
    return out


def merge_like_iamc(dfs: list[pd.DataFrame]) -> pd.DataFrame:
    """
    Merge multiple IAMC tables by using the first df's columns as canonical.
    """
    if not dfs:
        raise ValueError("No dataframes to merge.")
    base_cols = list(dfs[0].columns)
    parts = [dfs[0].copy()]
    for df in dfs[1:]:
        parts.append(align_to_base(df, base_cols))
    return pd.concat(parts, ignore_index=True)


def safe_output_path(original: Path, out_dir: Path) -> Path:
    """
    Create an output path that avoids overwriting originals (and name collisions).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    candidate = out_dir / original.name
    if candidate.exists():
        candidate = out_dir / f"{original.stem}__updated{original.suffix}"
    i = 2
    while candidate.exists():
        candidate = out_dir / f"{original.stem}__updated_{i}{original.suffix}"
        i += 1
    return candidate


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    subfolders = [p for p in ROOT_DIR.iterdir() if p.is_dir() and p.name != OUT_DIR.name]
    if not subfolders:
        print(f"[INFO] No subfolders found under {ROOT_DIR}")
        return

    for folder in subfolders:
        files = [p for p in folder.iterdir() if p.is_file()]
        excel_files = sorted([p for p in files if is_excel(p)])
        csv_files = sorted([p for p in files if is_csv(p)])

        if not excel_files and not csv_files:
            print(f"[SKIP] {folder.name}: no Excel or CSV files")
            continue

        # Read all CSVs in the folder
        csv_dfs = []
        for c in csv_files:
            df = read_csv_robust(c)
            if df is not None and not df.empty:
                csv_dfs.append(df)

        if excel_files:
            # Case A: Excel exists -> append CSV rows into each Excel's 'data' sheet
            if not csv_dfs:
                print(f"[SKIP] {folder.name}: Excel present but no readable CSVs to append")
                continue

            for xls_path in excel_files:
                base_df = read_excel_sheet(xls_path, SHEET_NAME)
                if base_df is None:
                    print(f"[SKIP] {folder.name}/{xls_path.name}: no sheet '{SHEET_NAME}'")
                    continue

                base_cols = list(base_df.columns)
                appended_parts = [base_df]

                for df in csv_dfs:
                    appended_parts.append(align_to_base(df, base_cols))

                combined = pd.concat(appended_parts, ignore_index=True)

                # Write new Excel (preserve other sheets by copying them)
                out_xlsx = safe_output_path(xls_path, OUT_DIR) if NEVER_OVERWRITE_ORIGINALS else xls_path

                try:
                    # Load original workbook sheets, replace 'data'
                    original = pd.ExcelFile(xls_path)
                    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                        for sh in original.sheet_names:
                            if sh == SHEET_NAME:
                                combined.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                            else:
                                tmp = pd.read_excel(original, sheet_name=sh)
                                tmp.to_excel(writer, sheet_name=sh, index=False)

                    print(f"[OK] {folder.name}: updated {xls_path.name} -> {out_xlsx.name} (rows={len(combined):,})")
                except Exception as e:
                    print(f"[WARN] {folder.name}/{xls_path.name}: failed writing updated Excel: {e}")

        else:
            # Case B: no Excel -> merge CSVs -> write one Excel with 'data' sheet
            if not csv_dfs:
                print(f"[SKIP] {folder.name}: no readable CSVs")
                continue

            merged = merge_like_iamc(csv_dfs)
            out_xlsx = OUT_DIR / f"{folder.name}.xlsx"

            # Avoid collision
            i = 2
            while out_xlsx.exists():
                out_xlsx = OUT_DIR / f"{folder.name}_{i}.xlsx"
                i += 1

            try:
                with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                    merged.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                print(f"[OK] {folder.name}: wrote {out_xlsx.name} (rows={len(merged):,}, csvs={len(csv_dfs)})")
            except Exception as e:
                print(f"[WARN] {folder.name}: failed writing Excel: {e}")


if __name__ == "__main__":
    main()


[SKIP] SSP2_narrow_act: Excel present but no readable CSVs to append
[SKIP] SSP2_narrow_act_26_tax: Excel present but no readable CSVs to append


In [35]:
def main2() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    FILTER_OUT_VARS = {
        'Material Stock|Chemicals|Plastics',
        'Material Outflow|Chemicals|Plastics',
        'Primary Energy|Non-Biomass Renewables|Geothermal',
        'Primary Energy|Non-Biomass Renewables|Hydro',
        'Primary Energy|Non-Biomass Renewables|Solar',
        'Primary Energy|Non-Biomass Renewables|Wind',
        'Secondary Energy|Gases|Natural Gas',
    }

    subfolders = [p for p in ROOT_DIR.iterdir() if p.is_dir() and p.name != OUT_DIR.name]
    if not subfolders:
        print(f"[INFO] No subfolders found under {ROOT_DIR}")
        return

    for folder in subfolders:
        files = [p for p in folder.iterdir() if p.is_file()]
        excel_files = sorted([p for p in files if is_excel(p)])
        csv_files = sorted([p for p in files if is_csv(p)])

        if not excel_files and not csv_files:
            print(f"[SKIP] {folder.name}: no Excel or CSV files")
            continue

        # Read all CSVs in the folder
        csv_dfs = []
        for c in csv_files:
            df = read_csv_robust(c)
            if df is not None and not df.empty:
                csv_dfs.append(df)

        if excel_files:
            if csv_dfs:
                # Case A: Excel exists + CSVs -> append CSV rows into each Excel's 'data' sheet
                for xls_path in excel_files:
                    base_df = read_excel_sheet(xls_path, SHEET_NAME)
                    if base_df is None:
                        print(f"[SKIP] {folder.name}/{xls_path.name}: no sheet '{SHEET_NAME}'")
                        continue

                    base_cols = list(base_df.columns)
                    appended_parts = [base_df]

                    for df in csv_dfs:
                        appended_parts.append(align_to_base(df, base_cols))

                    combined = pd.concat(appended_parts, ignore_index=True)

                    # Write new Excel (preserve other sheets by copying them)
                    out_xlsx = safe_output_path(xls_path, OUT_DIR) if NEVER_OVERWRITE_ORIGINALS else xls_path

                    try:
                        # Load original workbook sheets, replace 'data'
                        original = pd.ExcelFile(xls_path)
                        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                            for sh in original.sheet_names:
                                if sh == SHEET_NAME:
                                    combined.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                                else:
                                    tmp = pd.read_excel(original, sheet_name=sh)
                                    tmp.to_excel(writer, sheet_name=sh, index=False)

                        print(f"[OK] {folder.name}: updated {xls_path.name} -> {out_xlsx.name} (rows={len(combined):,})")
                    except Exception as e:
                        print(f"[WARN] {folder.name}/{xls_path.name}: failed writing updated Excel: {e}")
            else:
                # Case B: Excel files only (no CSVs) -> merge all Excel files together
                all_dfs = []
                for xls_path in excel_files:
                    try:
                        df = read_excel_sheet(xls_path, SHEET_NAME)
                        if df is not None:
                            # Normalize column names: convert to string first (handles integer year columns), then lowercase
                            df.columns = [str(c).lower() for c in df.columns]
                            all_dfs.append(df)
                            print(f"[READ] {folder.name}/{xls_path.name}: {len(df)} rows, columns: {list(df.columns)[:8]}")
                        else:
                            print(f"[SKIP] {folder.name}/{xls_path.name}: no sheet '{SHEET_NAME}'")
                    except Exception as e:
                        print(f"[WARN] {folder.name}/{xls_path.name}: failed reading: {e}")

                if all_dfs:
                    # Use the first DataFrame's columns as the canonical structure
                    base_cols = list(all_dfs[0].columns)
                    
                    # Align all other DataFrames to match the base structure
                    aligned_dfs = [all_dfs[0]]
                    for i, df in enumerate(all_dfs[1:], 1):
                        aligned_df = align_to_base(df, base_cols)
                        aligned_dfs.append(aligned_df)
                    
                    # Now concatenate with aligned columns
                    merged = pd.concat(aligned_dfs, ignore_index=True)

                    before_count = len(merged)
                    merged = merged[~merged['variable'].isin(FILTER_OUT_VARS)]
                    after_count = len(merged)
                    if before_count != after_count:
                        print(f"[INFO] {folder.name}: Filtered out {before_count - after_count} rows matching blocked variables")

                    
                    print(f"[INFO] {folder.name}: Merged columns: {list(merged.columns)[:10]}...")
                    print(f"[INFO] {folder.name}: Total rows={len(merged)}, Total columns={len(merged.columns)}")
                    
                    out_xlsx = OUT_DIR / f"{folder.name}_combined.xlsx"

                    # Avoid collision
                    i = 2
                    while out_xlsx.exists():
                        out_xlsx = OUT_DIR / f"{folder.name}_combined_{i}.xlsx"
                        i += 1

                    try:
                        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                            merged.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                        print(f"[OK] {folder.name}: merged {len(all_dfs)} Excel files -> {out_xlsx.name} (rows={len(merged):,})")
                    except Exception as e:
                        print(f"[WARN] {folder.name}: failed writing merged Excel: {e}")
                else:
                    print(f"[SKIP] {folder.name}: no readable Excel sheets")

        else:
            # Case C: CSV files only (no Excel) -> merge CSVs -> write one Excel with 'data' sheet
            if not csv_dfs:
                print(f"[SKIP] {folder.name}: no readable CSVs")
                continue

            merged = pd.concat(csv_dfs, ignore_index=True)
            out_xlsx = OUT_DIR / f"{folder.name}.xlsx"

            # Avoid collision
            i = 2
            while out_xlsx.exists():
                out_xlsx = OUT_DIR / f"{folder.name}_{i}.xlsx"
                i += 1

            try:
                with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                    merged.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                print(f"[OK] {folder.name}: wrote {out_xlsx.name} (rows={len(merged):,}, csvs={len(csv_dfs)})")
            except Exception as e:
                print(f"[WARN] {folder.name}: failed writing Excel: {e}")

if __name__ == "__main__":
    main2()


[READ] SSP2_narrow_act/SSP2_CP_narrow_act.xlsx: 8572 rows, columns: ['model', 'scenario', 'region', 'variable', 'unit', '2005', '2010', '2015']
[READ] SSP2_narrow_act/SSP2_narrow_act.xlsx: 19344 rows, columns: ['model', 'scenario', 'region', 'variable', 'unit', '2005', '2010', '2015']
[INFO] SSP2_narrow_act: Filtered out 187 rows matching blocked variables
[INFO] SSP2_narrow_act: Merged columns: ['model', 'scenario', 'region', 'variable', 'unit', '2005', '2010', '2015', '2020', '2025']...
[INFO] SSP2_narrow_act: Total rows=27729, Total columns=20
[OK] SSP2_narrow_act: merged 2 Excel files -> SSP2_narrow_act_combined.xlsx (rows=27,729)
[READ] SSP2_narrow_act_26_tax/SSP2_CP_narrow_26_tax_mat.xlsx: 8545 rows, columns: ['model', 'scenario', 'region', 'variable', 'unit', '2005', '2010', '2015']
[READ] SSP2_narrow_act_26_tax/SSP2_narrow_act_26_tax.xlsx: 19344 rows, columns: ['model', 'scenario', 'region', 'variable', 'unit', '2005', '2010', '2015']
[INFO] SSP2_narrow_act_26_tax: Filtered out

In [28]:
import pandas as pd
from pathlib import Path

base = Path(r"c:\Users\5982758\repos\data\April 2026 runs")

old_file = base / "other scenarios" / "files last round" / "1770815965538-SSP2_combined.xlsx"
new_file = base / "combined" / "_updated_excels" / "SSP2_combined.xlsx"

old_df = pd.read_excel(old_file, sheet_name="data")
new_df = pd.read_excel(new_file, sheet_name="data")

# normalize column name
old_var_col = "Variable" if "Variable" in old_df.columns else "variable"
new_var_col = "Variable" if "Variable" in new_df.columns else "variable"

old_vars = set(old_df[old_var_col].dropna().unique())
new_vars = set(new_df[new_var_col].dropna().unique())

missing_in_new = sorted(old_vars - new_vars)
added_in_new = sorted(new_vars - old_vars)

print(f"Old file: {old_var_col} column, {len(old_vars)} unique variables")
print(f"New file: {new_var_col} column, {len(new_vars)} unique variables")
print(f"Common: {len(old_vars & new_vars)}")
print(f"Missing in new: {len(missing_in_new)}")
print(f"Added in new: {len(added_in_new)}")


Old file: variable column, 971 unique variables
New file: variable column, 1056 unique variables
Common: 971
Missing in new: 0
Added in new: 85


In [17]:
# Just the added sectors
sectors_add = set()
for v in added_in_new:
    p = v.split("|")
    if len(p) >= 3:
        sectors_add.add("|".join(p[2:]))
    elif len(p) == 2:
        sectors_add.add(p[1])

print("Sectors in ADDED vars:")
for s in sorted(sectors_add):
    print(f"  {s}")

# Category counts for added
from collections import Counter
add_cats = Counter(v.split("|")[0] for v in added_in_new)
print("\nAdded by category:")
for c, n in add_cats.most_common():
    print(f"  {c}: {n}")


Sectors in ADDED vars:
  Biomass
  Biomass|w/ CCS
  Biomass|w/o CCS
  Coal
  Coal|w/ CCS
  Coal|w/o CCS
  Electricity
  Electricity|w/ CCS
  Electricity|w/o CCS
  Fossil
  Fossil|w/ CCS
  Fossil|w/o CCS
  Gas
  Gases
  Gas|w/ CCS
  Gas|w/o CCS
  Geothermal
  Heat
  Hydro
  Hydrogen
  Industrial Processes|Iron and Steel
  Iron and Steel|Steel|Primary|Blast Furnace and Basic Oxygen Furnace BAT
  Iron and Steel|Steel|Primary|Blast Furnace and Basic Oxygen Furnace Standard
  Iron and Steel|Steel|Primary|Blast Furnace and Basic Oxygen Furnace Standard w/ CCS
  Iron and Steel|Steel|Primary|Direct Reduced Iron and Electric Arc Furnace
  Iron and Steel|Steel|Primary|Direct Reduced Iron and Electric Arc Furnace w/ CCS
  Iron and Steel|Steel|Primary|Electrowinning and Electric Arc Furnace
  Iron and Steel|Steel|Primary|Hydrogen-Based Direct Reduced Iron and Electric Arc Furnace
  Iron and Steel|Steel|Primary|Smelting Reduction and Basic Oxygen Furnace
  Iron and Steel|Steel|Primary|Smelting Redu

In [32]:
import pandas as pd
from pathlib import Path

out = Path(r"c:\Users\5982758\repos\data\April 2026 runs\combined\_updated_excels")

df_nsc = pd.read_excel(out / "SSP2_narrow_slow_close_combined.xlsx", sheet_name="data")
df_nsc26 = pd.read_excel(out / "SSP2_narrow_slow_close_26_tax_combined.xlsx", sheet_name="data")

vars_nsc = set(df_nsc["variable"].dropna().unique())
vars_nsc26 = set(df_nsc26["variable"].dropna().unique())

only_base = sorted(vars_nsc - vars_nsc26)
only_26 = sorted(vars_nsc26 - vars_nsc)

print(f"SSP2_narrow_slow_close: {len(vars_nsc)} vars, {len(df_nsc)} rows")
print(f"SSP2_narrow_slow_close_26_tax: {len(vars_nsc26)} vars, {len(df_nsc26)} rows")
print(f"Common: {len(vars_nsc & vars_nsc26)}")

print(f"\n--- In base but NOT in 26_tax ({len(only_base)}) ---")
for v in only_base:
    print(f"  {v}")

print(f"\n--- In 26_tax but NOT in base ({len(only_26)}) ---")
for v in only_26:
    print(f"  {v}")


SSP2_narrow_slow_close: 1056 vars, 27729 rows
SSP2_narrow_slow_close_26_tax: 1055 vars, 27702 rows
Common: 1055

--- In base but NOT in 26_tax (1) ---
  Price|Carbon

--- In 26_tax but NOT in base (0) ---
